# THADAM — MPLADS AI Monitoring · run it free, in the cloud

**SIH 2026 · PS 26102 (MoSPI) · Team Morior Invictus**

This runs the complete site — 210,993 works, 37,705 flagged, the visit planner, the case
files, the assistant — on a free Google Colab machine, and gives you a public link anyone
can open. No credit card, no server to rent, nothing to install on your laptop.

**How to use it**

1. `Runtime -> Run all` (or run the cells in order).
2. Wait for **Cell 2** to print a `https://....trycloudflare.com` link.
3. Share that link. It works until you close this tab.

Photographs are read by **RapidOCR**, and the screen says so. Surya — the reader your
laptop uses — needs a GPU *and* a 10-minute compile here, so it is an optional extra in
Cell 3 rather than part of the normal run.

Everything is pulled from the project's public Hugging Face repository
[`kannanyuvaraj/Mplads`](https://huggingface.co/spaces/kannanyuvaraj/Mplads), so this notebook needs no upload.

## Cell 1 — fetch the project and install what it needs (~3 minutes)

In [ ]:
# The code, the built frontend and the pre-computed artifacts (~155 MB) come from the
# project's public Hugging Face repo, so nothing has to be uploaded from your machine.
!pip -q install "huggingface_hub>=0.34"

import os, subprocess, sys
from huggingface_hub import snapshot_download

PROJECT = snapshot_download(repo_id="kannanyuvaraj/Mplads", repo_type="space")
print("project at:", PROJECT)

# Serving needs far less than developing: a dataframe library, a web server, and the
# optional photo reader. No scikit-learn, no lifelines — the models already ran.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                os.path.join(PROJECT, "requirements-serve.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "rapidocr-onnxruntime>=1.4"], check=False)

os.environ["PYTHONPATH"] = os.path.join(PROJECT, "src")
os.environ["MPLADS_OCR_ENGINES"] = "rapidocr"   # Cell 3 switches this to Surya on a GPU
print("ready")

## Cell 2 — start the site and get a public link

In [ ]:
import os, re, subprocess, sys, time, urllib.request

PORT = 8020
LOG, TUNNEL_LOG = "/content/mplads.log", "/content/tunnel.log"

# Cloudflare's quick tunnel gives a public https address for a process on this machine.
# No account, no card. The address is new every run and lives as long as this notebook.
if not os.path.exists("/content/cloudflared"):
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /content/cloudflared

api = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "mplads.api.app:app",
     "--host", "0.0.0.0", "--port", str(PORT), "--workers", "1"],
    cwd=PROJECT, stdout=open(LOG, "wb"), stderr=subprocess.STDOUT,
    env={**os.environ, "PYTHONPATH": os.path.join(PROJECT, "src")})

def wait_for(url, timeout=420):
    """The corpus loads into memory once at startup; that is what we are waiting on."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=3) as r:
                return r.read().decode()
        except Exception:
            time.sleep(3)
    raise SystemExit("the site did not start — see " + LOG)

print("starting the site …")
print("health:", wait_for(f"http://127.0.0.1:{PORT}/api/health"))

tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url",
                           f"http://localhost:{PORT}", "--no-autoupdate"],
                          stdout=open(TUNNEL_LOG, "wb"), stderr=subprocess.STDOUT)

public = None
for _ in range(60):
    time.sleep(2)
    text = open(TUNNEL_LOG, encoding="utf-8", errors="replace").read()
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
    if found:
        public = found.group(0)
        break

print("\n" + "=" * 68)
print("  YOUR PUBLIC LINK:", public or "not ready — re-run this cell")
print("=" * 68)
print("\nSign in with  auditor / mplads2026  to record a site visit.")
print("The link stops working when this notebook is closed.")

## Cell 3 — optional, slow: Surya OCR on the GPU

**Skip this unless you specifically want to demonstrate Surya.** Photographs are already
read by RapidOCR without it.

llama.cpp ships no ready-made Linux CUDA binary, so using the GPU means compiling it here:
about 10 minutes, plus a 1.5 GB model download. Set **Runtime → Change runtime type → T4
GPU** first. On a CPU runtime Surya would take minutes per photograph, so this cell
deliberately refuses to enable it there.

In [ ]:
import json, os, pathlib, shutil, subprocess, sys, time, urllib.request

if not shutil.which("nvidia-smi"):
    print("No GPU on this runtime — leaving photo reading with RapidOCR, which is the right")
    print("choice here: Surya on a CPU is minutes per board.")
    print("To use it: Runtime -> Change runtime type -> T4 GPU, then run the cells again.")
else:
    print("Building llama.cpp with CUDA. This takes about 10 minutes; it is the only way to")
    print("run Surya on the GPU, because the project publishes no Linux CUDA binary.")
    if not os.path.exists("/content/llama.cpp"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/ggml-org/llama.cpp", "/content/llama.cpp"], check=True)
    subprocess.run(["cmake", "-B", "/content/llama.cpp/build", "-S", "/content/llama.cpp",
                    "-DGGML_CUDA=ON", "-DLLAMA_CURL=OFF", "-DCMAKE_BUILD_TYPE=Release"],
                   check=True, capture_output=True)
    subprocess.run(["cmake", "--build", "/content/llama.cpp/build", "--config", "Release",
                    "-j", "--target", "llama-server"], check=True, capture_output=True)
    binary = "/content/llama.cpp/build/bin/llama-server"
    print("built:", binary, "exists:", os.path.exists(binary))

    from huggingface_hub import hf_hub_download
    model = hf_hub_download("datalab-to/surya-ocr-2-gguf", "surya-2.gguf")
    mmproj = hf_hub_download("datalab-to/surya-ocr-2-gguf", "surya-2-mmproj.gguf")

    # The site treats a model as usable only when a manifest says the files are complete —
    # a half-downloaded model has a valid header and would fail deep inside startup.
    manifest = pathlib.Path(model).parent / "manifest.json"
    manifest.write_text(json.dumps({"repo": "datalab-to/surya-ocr-2-gguf", "files": {
        pathlib.Path(model).name: {"size": os.path.getsize(model)},
        pathlib.Path(mmproj).name: {"size": os.path.getsize(mmproj)}}}), encoding="utf-8")

    os.environ.update({"MPLADS_OCR_ENGINES": "surya,rapidocr",
                       "MPLADS_LLAMA_SERVER": binary,
                       "MPLADS_SURYA_GGUF_DIR": str(pathlib.Path(model).parent)})

    print("restarting the site with Surya enabled …")
    api.terminate()
    time.sleep(3)
    api = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "mplads.api.app:app",
         "--host", "0.0.0.0", "--port", str(PORT), "--workers", "1"],
        cwd=PROJECT, stdout=open(LOG, "wb"), stderr=subprocess.STDOUT, env=dict(os.environ))
    print("health:", wait_for(f"http://127.0.0.1:{PORT}/api/health"))
    with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/ocr/status", timeout=60) as r:
        reader = json.load(r)["photographs"]["primary"]
    print("photo reader is now:", reader, "| your link is unchanged:", public)

## Cell 4 — keep it awake while you demo, and see what the site is doing

In [ ]:
# Colab disconnects an idle notebook. This prints a heartbeat so the session stays active;
# stop it (the square button) whenever you like — the site keeps running either way.
import time, urllib.request, datetime

while True:
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/health", timeout=5) as r:
            health = r.read().decode()
    except Exception as exc:
        health = "not responding: " + str(exc)
    print(datetime.datetime.now().strftime("%H:%M:%S"), health, "|", public)
    time.sleep(120)

---

### If something goes wrong

- **No link printed** — re-run Cell 2; Cloudflare occasionally refuses the first attempt.
- **Site not starting** — open `/content/mplads.log`.
- **Pages load but figures are missing** — the artifacts did not download; re-run Cell 1.

### What is honest about this deployment

- It is the real system: the same optimiser, the same case files, the same 210,993 works.
- **The link is temporary** and dies with the notebook. For a permanent address the project
  needs a paid Space, or a host with about 1 GB of memory.
- Site-visit reports recorded here are lost when the runtime stops.
- Everything the site shows is an **investigation lead with evidence — never a fraud
  verdict**, and a person decides every action.